In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import sqlalchemy
import json
from google.colab import files
import warnings
warnings.filterwarnings('ignore')


In [2]:
file = "https://docs.google.com/spreadsheets/d/1peY6zuMhmlez6nWbdwmWCnO1uOxpojm5sCne5B5OLfo/export?format=csv"

data = pd.read_csv(file)
data.head()

,kategory,kelas,cover_buku,judul_bab,subjudul_bab,abstrak,use_case,materi,soal,link
0,SMA,X,Matematika untuk SMA/MA/SMK/MAK Kelas X (Edisi...,Eksponen,- Eksponen\n- Sifat-sifat Eksponen\n- Fungsi E...,Bab ini berfokus pada pemahaman mendalam tenta...,1. Pertumbuhan Penduduk: Memprediksi jumlah pe...,- Definisi Eksponen: Eksponen adalah bentuk pe...,1. Hasil dari $(2^3 \times 2^2)^2$ adalah...\n...,https://buku.kemendikdasmen.go.id/katalog/Mate...
1,SMA,X,Matematika untuk SMA/MA/SMK/MAK Kelas X (Edisi...,Barisan dan Deret,- Barisan (Aritmetika dan Geometri)\r\n- Deret...,Bab ini membahas pola bilangan yang tersusun s...,1. Tabungan Bank: Menghitung saldo akhir tabun...,- Barisan: Penjelasan mengenai Barisan Aritmet...,"1. Suku ke-10 dari barisan aritmetika 2, 5, 8,...",https://buku.kemendikdasmen.go.id/katalog/mate...
2,SMA,X,Matematika untuk SMA/MA/SMK/MAK Kelas X (Edisi...,Perbandingan Trigonometri,- Perbandingan Trigonometri\r\n- Pemanfaatan P...,Bab ini memperkenalkan konsep dasar perbanding...,1. Navigasi Pelayaran: Menentukan jarak kapal ...,- Perbandingan Trigonometri: Pembahasan mengen...,"1. Pada segitiga siku-siku, perbandingan sisi ...",https://buku.kemendikdasmen.go.id/katalog/mate...
3,SMA,X,Matematika untuk SMA/MA/SMK/MAK Kelas X (Edisi...,Sistem Persamaan dan Pertidaksamaan Linear,- Sistem Persamaan Linear (Dua dan Tiga Variab...,Bab ini membahas cara mencari solusi dari bebe...,1. Analisis Bisnis: Menghitung jumlah kombinas...,- Sistem Persamaan Linear: Berfokus pada Siste...,1. Himpunan penyelesaian dari $x + y = 5$ dan ...,https://buku.kemendikdasmen.go.id/katalog/mate...
4,SMA,X,Matematika untuk SMA/MA/SMK/MAK Kelas X (Edisi...,Persamaan dan Fungsi Kuadrat,- Persamaan Kuadrat\r\n- Fungsi Kuadrat,Bab ini membahas tentang persamaan polinomial ...,1. Gerak Parabola: Menghitung lintasan bola ya...,- Persamaan Kuadrat: Bentuk umum $ax^2 + bx + ...,1. Akar-akar dari persamaan kuadrat $x^2 - 5x ...,https://buku.kemendikdasmen.go.id/katalog/mate...


In [3]:
data_bersih = data.copy()

# Bersihin Kolom yang Berisi List

In [4]:
def rapihkan_list_data(text):
    if not isinstance(text, str):
        return text

    text = text.replace('\r\n', '\n')
    items = text.split('\n')
    cleaned_items = []
    for item in items:
        clean_item = item.strip()
        clean_item = re.sub(r'^\d+\.\s+|^-\s+|^\*\s+', '', clean_item)
        if clean_item:
            cleaned_items.append(clean_item)
    return cleaned_items

kolom_berlist = ['subjudul_bab', 'use_case', 'materi', 'soal']
for col in kolom_berlist:
    data_bersih[col] = data_bersih[col].apply(rapihkan_list_data)

display(data_bersih[kolom_berlist].head())

,subjudul_bab,use_case,materi,soal
0,"[Eksponen, Sifat-sifat Eksponen, Fungsi Ekspon...",[Pertumbuhan Penduduk: Memprediksi jumlah pend...,[Definisi Eksponen: Eksponen adalah bentuk per...,"[Hasil dari $(2^3 \times 2^2)^2$ adalah..., a...."
1,"[Barisan (Aritmetika dan Geometri), Deret (Ari...",[Tabungan Bank: Menghitung saldo akhir tabunga...,[Barisan: Penjelasan mengenai Barisan Aritmeti...,"[Suku ke-10 dari barisan aritmetika 2, 5, 8, 1..."
2,"[Perbandingan Trigonometri, Pemanfaatan Perban...",[Navigasi Pelayaran: Menentukan jarak kapal ke...,[Perbandingan Trigonometri: Pembahasan mengena...,"[Pada segitiga siku-siku, perbandingan sisi de..."
3,[Sistem Persamaan Linear (Dua dan Tiga Variabe...,[Analisis Bisnis: Menghitung jumlah kombinasi ...,[Sistem Persamaan Linear: Berfokus pada Sistem...,[Himpunan penyelesaian dari $x + y = 5$ dan $x...
4,"[Persamaan Kuadrat, Fungsi Kuadrat]",[Gerak Parabola: Menghitung lintasan bola yang...,[Persamaan Kuadrat: Bentuk umum $ax^2 + bx + c...,[Akar-akar dari persamaan kuadrat $x^2 - 5x + ...


# Bersihin Simbol LaTeX

In [5]:
def bersihin_latex(text):
    if not isinstance(text, str):
        return text

    text = re.sub(r'\$(.*?)\$', r'\1', text)
    text = re.sub(r'\$\$(.*?)\$\$', r'\1', text)

    # Replace common LaTeX commands with more readable forms
    text = text.replace('\\sqrt', 'sqrt')   # Replace \sqrt with sqrt
    text = text.replace('\\times', 'x')     # Replace \times with x (multiplication)
    text = text.replace('\\cdot', '*')      # Replace \cdot with *
    text = text.replace('\\frac{', '/')     # Replace \frac{a}{b} (simplified)
    text = text.replace('}', '')              # Remove closing brace for \frac (simplistic)
    text = text.replace('\\ldots', '...')   # Replace \ldots with ...
    text = text.replace('\\ge', '>=')       # Replace \ge with >=
    text = text.replace('\\le', '<=')       # Replace \le with <=
    text = text.replace('\\pi', 'pi')       # Replace \pi with pi
    text = text.replace('\\alpha', 'alpha') # Replace \alpha with alpha
    text = text.replace('\\beta', 'beta')   # Replace \beta with beta
    text = text.replace('\\gamma', 'gamma') # Replace \gamma with gamma
    text = text.replace('\\div', '/')       # Replace \div with /

    text = re.sub(r'([+\-*/=<>])\s+', r'\1', text)
    text = re.sub(r'\s+([+\-*/=<>])', r'\1', text)

    return text

for col in ['materi', 'soal']:
    data_bersih[col] = data_bersih[col].apply(lambda x: [bersihin_latex(item) for item in x] if isinstance(x, list) else bersihin_latex(x))

display(data_bersih[['materi', 'soal']].head())

,materi,soal
0,[Definisi Eksponen: Eksponen adalah bentuk per...,"[Hasil dari (2^3 x 2^2)^2 adalah..., a. 2^7, b..."
1,[Barisan: Penjelasan mengenai Barisan Aritmeti...,"[Suku ke-10 dari barisan aritmetika 2, 5, 8, 1..."
2,[Perbandingan Trigonometri: Pembahasan mengena...,"[Pada segitiga siku-siku, perbandingan sisi de..."
3,[Sistem Persamaan Linear: Berfokus pada Sistem...,[Himpunan penyelesaian dari x+y=5 dan x-y=1 ad...
4,[Persamaan Kuadrat: Bentuk umum ax^2+bx+c=0. M...,[Akar-akar dari persamaan kuadrat x^2-5x+6=0 a...


In [6]:
data_bersih.head()

,kategory,kelas,cover_buku,judul_bab,subjudul_bab,abstrak,use_case,materi,soal,link
0,SMA,X,Matematika untuk SMA/MA/SMK/MAK Kelas X (Edisi...,Eksponen,"[Eksponen, Sifat-sifat Eksponen, Fungsi Ekspon...",Bab ini berfokus pada pemahaman mendalam tenta...,[Pertumbuhan Penduduk: Memprediksi jumlah pend...,[Definisi Eksponen: Eksponen adalah bentuk per...,"[Hasil dari (2^3 x 2^2)^2 adalah..., a. 2^7, b...",https://buku.kemendikdasmen.go.id/katalog/Mate...
1,SMA,X,Matematika untuk SMA/MA/SMK/MAK Kelas X (Edisi...,Barisan dan Deret,"[Barisan (Aritmetika dan Geometri), Deret (Ari...",Bab ini membahas pola bilangan yang tersusun s...,[Tabungan Bank: Menghitung saldo akhir tabunga...,[Barisan: Penjelasan mengenai Barisan Aritmeti...,"[Suku ke-10 dari barisan aritmetika 2, 5, 8, 1...",https://buku.kemendikdasmen.go.id/katalog/mate...
2,SMA,X,Matematika untuk SMA/MA/SMK/MAK Kelas X (Edisi...,Perbandingan Trigonometri,"[Perbandingan Trigonometri, Pemanfaatan Perban...",Bab ini memperkenalkan konsep dasar perbanding...,[Navigasi Pelayaran: Menentukan jarak kapal ke...,[Perbandingan Trigonometri: Pembahasan mengena...,"[Pada segitiga siku-siku, perbandingan sisi de...",https://buku.kemendikdasmen.go.id/katalog/mate...
3,SMA,X,Matematika untuk SMA/MA/SMK/MAK Kelas X (Edisi...,Sistem Persamaan dan Pertidaksamaan Linear,[Sistem Persamaan Linear (Dua dan Tiga Variabe...,Bab ini membahas cara mencari solusi dari bebe...,[Analisis Bisnis: Menghitung jumlah kombinasi ...,[Sistem Persamaan Linear: Berfokus pada Sistem...,[Himpunan penyelesaian dari x+y=5 dan x-y=1 ad...,https://buku.kemendikdasmen.go.id/katalog/mate...
4,SMA,X,Matematika untuk SMA/MA/SMK/MAK Kelas X (Edisi...,Persamaan dan Fungsi Kuadrat,"[Persamaan Kuadrat, Fungsi Kuadrat]",Bab ini membahas tentang persamaan polinomial ...,[Gerak Parabola: Menghitung lintasan bola yang...,[Persamaan Kuadrat: Bentuk umum ax^2+bx+c=0. M...,[Akar-akar dari persamaan kuadrat x^2-5x+6=0 a...,https://buku.kemendikdasmen.go.id/katalog/mate...


# Download CSV

In [7]:
nama_file_csv = 'data_bersih.csv'
data_bersih.to_csv(nama_file_csv, index=False)

# Download the CSV file to your local machine
files.download(nama_file_csv)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Download SQL

In [9]:
def df_to_sql_insert(df, table_name, file_name='output.sql'):
    """Converts a pandas DataFrame to SQL INSERT statements and saves to a file."""
    sql_statements = []
    # Escape table name
    table_name_esc = sqlalchemy.sql.quoted_name(table_name, quote=True)

    # Generate column names string for INSERT statement
    columns_str = ', '.join([sqlalchemy.sql.quoted_name(col, quote=True).replace("'", '"') for col in df.columns])

    for index, row in df.iterrows():
        values = []
        for item in row:
            if isinstance(item, list):
                # Convert list to a string representation that SQL can store, e.g., JSON string
                values.append(f"'{str(item).replace("\'", "''")}'") # Escape single quotes for SQL
            elif isinstance(item, str):
                values.append(f"'{item.replace("\'", "''")}'") # Escape single quotes for SQL
            elif pd.isna(item):
                values.append('NULL')
            else:
                values.append(str(item))
        values_str = ', '.join(values)
        sql_statements.append(f"INSERT INTO {table_name_esc} ({columns_str}) VALUES ({values_str});")

    with open(file_name, 'w', encoding='utf-8') as f:
        for stmt in sql_statements:
            f.write(stmt + '\n')
    return file_name

# Define the table name for the SQL file
table_name = 'data_bersih_table'
sql_file_name = 'data_sql.sql'

# Generate SQL statements and save to file
df_to_sql_insert(data_bersih, table_name, sql_file_name)

# Download the SQL file
files.download(sql_file_name)

print(f"File '{sql_file_name}' telah berhasil dibuat dan siap diunduh.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File 'data_sql.sql' telah berhasil dibuat dan siap diunduh.


# Download JSON

In [11]:
json_file_name = 'data_json.json'
# Convert DataFrame to JSON format.
# Using orient='records' creates a list of JSON objects, where each object is a row.
# indent=4 makes the JSON output human-readable.
data_bersih.to_json(json_file_name, orient='records', indent=4)

# Download the JSON file
files.download(json_file_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>